In [ ]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import array_to_img

# Load the best model saved by ModelCheckpoint
best_model = load_model('best_model.h5')


In [ ]:

# Create the directory for saving output images if it doesn't exist
output_dir = './images/val_outputs'
os.makedirs(output_dir, exist_ok=True)

# Function to predict and save the output masks
def predict_and_save(validation_dir, target_size=(512, 512), output_dir='./images/val_outputs'):
    # List of validation images
    validation_images = sorted(os.listdir(validation_dir))
    
    for img_name in validation_images:
        # Load and preprocess the image
        img_path = os.path.join(validation_dir, img_name)
        img = load_img(img_path, target_size=target_size)
        img = img_to_array(img) / 255.0  # Normalize the image
        img = np.expand_dims(img, axis=0)  # Add batch dimension
        
        # Predict the mask for this image
        pred_mask = best_model.predict(img)
        
        # Get the class with the highest probability for each pixel
        pred_mask_class = np.argmax(pred_mask, axis=-1)  # Convert to class labels (shape: (height, width))

        # Convert the predicted mask back to an image (grayscale)
        pred_mask_img = array_to_img(pred_mask_class[0])  # Convert numpy array to image

        # Save the predicted mask to the output directory
        output_path = os.path.join(output_dir, f"pred_{img_name}")
        pred_mask_img.save(output_path)  # Save as an image file

        print(f"Saved predicted mask for {img_name} to {output_path}")


In [ ]:
# Example usage
validation_dir = './images/satellites'
predict_and_save(validation_dir, output_dir=output_dir)

print(f"Predictions saved to {output_dir}")